# Feature Engineering Analysis

**Purpose:** Investigate potential new features that could improve model performance, based on insights from descriptive and inferential analysis.

**Tasks:** 

1. Create new features by combining existing ones (e.g., date-based features like "day of the week").
2. Encode categorical variables: Label encoding, One-Hot encoding, or similar techniques.
3. Feature scaling: Min-Max scaling, Standardization.
4. Feature selection techniques (e.g., based on correlation matrix, mutual information, feature importance).

In [2]:
import json

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats
import statsmodels.api as sm
import statsmodels.stats.api as sms

pd.set_option('display.max_columns', None)

In [3]:
# reading data
df = pd.read_csv(r"D:\Doccuments\GitHub\End-to-End-MLOps-Pipeline\data\raw\fraud test.csv")
df.shape

(555719, 23)

In [4]:
data = df[df.columns[1:]].copy()
data.head(3)

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,21/06/2020 12:14,2.291160e+15,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,Columbia,SC,29209,33.9659,-80.9355,333497,Mechanical engineer,19/03/1968,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0
1,21/06/2020 12:14,3.573030e+15,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,Altonah,UT,84002,40.3207,-110.4360,302,"Sales professional, IT",17/01/1990,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0
2,21/06/2020 12:14,3.598220e+15,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,Ashley,Lopez,F,9333 Valentine Point,Bellmore,NY,11710,40.6729,-73.5365,34496,"Librarian, public",21/10/1970,c81755dbbbea9d5c77f094348a7579be,1371816893,40.495810,-74.196111,0


In [5]:
cols_div_path = r"D:\Doccuments\GitHub\End-to-End-MLOps-Pipeline\notebooks\metadata\Credit_cols_classified.json"
with open(cols_div_path,'r') as file:
    cols_division = json.load(file)
cols_division

{'target_col': 'is_fraud',
 'uniq_cols': ['cc_num', 'trans_num'],
 'num_cols': ['amt', 'city_pop'],
 'cat_cols': ['merchant',
  'category',
  'first',
  'last',
  'gender',
  'street',
  'city',
  'state',
  'job'],
 'loc_cols': ['long', 'lat', 'merch_lat', 'merch_long', 'zip'],
 'time_cols': ['trans_date_trans_time', 'dob', 'unix_time']}

In [6]:
data["amt_per_pop"] = data["amt"] / (data["city_pop"] + 1e-6)
data["log_amt_per_pop"] = np.log1p(data["amt"]) - np.log1p(data["city_pop"])

The feature `log_amt_per_pop` helps capture:

- High transaction amount in small population → suspicious
- Same amount in large city → less suspicious

In [7]:
data.groupby(by = "is_fraud")["log_amt_per_pop"].describe().T

is_fraud,0,1
count,553574.000000,2145.000000
mean,-4.831561,-2.826417
std,2.783637,2.658438
min,-14.184395,-12.233963
25%,-6.640328,-4.592013
50%,-4.432446,-2.374136
75%,-2.776941,-0.800687
max,4.122553,3.559684


`first`, `last` and `street` can be dropped as it will not give any kind of usefull information.

In [8]:
data["log_amt"] = np.log1p(data["amt"])
data["amt_zscore"] = (data["amt"] - data["amt"].mean()) / data["amt"].std()

In [9]:
data["amt_user_mean"] = data.groupby("cc_num")["amt"].transform("mean")
data["amt_user_std"] = data.groupby("cc_num")["amt"].transform("std")

data["amt_user_zscore"] = (data["amt"] - data["amt_user_mean"]) / (data["amt_user_std"] + 1e-6)

In [10]:
# Transaction count features:
data["user_txn_count"] = data.groupby("cc_num")["amt"].transform("count")
data["user_avg_amt"] = data.groupby("cc_num")["amt"].transform("mean")

In [11]:
# Merchant interaction:
data["user_merchant_count"] = data.groupby(["cc_num", "merchant"])["amt"].transform("count")

In [12]:
data["trans_date_trans_time"] = pd.to_datetime(data["trans_date_trans_time"])
data["dob"] = pd.to_datetime(data["dob"])

C:\Users\HP\AppData\Local\Temp\ipykernel_17108\336223865.py:1: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  data["trans_date_trans_time"] = pd.to_datetime(data["trans_date_trans_time"])
C:\Users\HP\AppData\Local\Temp\ipykernel_17108\336223865.py:2: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  data["dob"] = pd.to_datetime(data["dob"])


In [13]:
data["hour"] = data["trans_date_trans_time"].dt.hour
data["day"] = data["trans_date_trans_time"].dt.day
data["month"] = data["trans_date_trans_time"].dt.month
data["weekday"] = data["trans_date_trans_time"].dt.weekday
data["is_weekend"] = data["weekday"].isin([5,6]).astype(int)

In [14]:
data["age"] = (data["trans_date_trans_time"] - data["dob"]).dt.days // 365

In [15]:
# Time gap between transactions
data = data.sort_values(["cc_num", "trans_date_trans_time"])

data["prev_time"] = data.groupby("cc_num")["trans_date_trans_time"].shift(1)
data["time_diff"] = (data["trans_date_trans_time"] - data["prev_time"]).dt.total_seconds()

### Feature Encoding

In [16]:
freq = data["merchant"].value_counts()
data["merchant_freq"] = data["merchant"].map(freq)

freq = data["job"].value_counts()
data["job_freq"] = data["job"].map(freq)

freq = data["city"].value_counts()
data["city_freq"] = data["city"].map(freq)

In [ ]:
data = pd.get_dummies(data, columns=["gender", "category", "state"], drop_first=True,dtype=int)
data.head()

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 555719 entries, 157 to 553883
Columns: 103 entries, trans_date_trans_time to state_WY
dtypes: datetime64[ns](3), float64(15), int32(4), int64(74), object(7)
memory usage: 432.5+ MB


In [ ]:
import os
path = r"D:\Doccuments\GitHub\End-to-End-MLOps-Pipeline\data\precessed"
file = os.path.join(path,"CC_Data_feature_engineered.csv")
os.makedirs(file,exist_ok=True)
data.to_csv(index = False)

NameError: name 'data' is not defined